<a href="https://colab.research.google.com/github/TeamAsmaa/phishing-link-analyzer/blob/main/notebooks/phishing_link_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
!pip install -q langchain-google-genai langchain langchain-core pydantic requests

In [83]:
import base64
import requests
from typing import List

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

In [84]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
VT_API_KEY = userdata.get("VT_API_KEY")

MODEL_NAME = "gemini-2.5-flash"

In [85]:
llm = ChatGoogleGenerativeAI(
    google_api_key=GOOGLE_API_KEY,
    model=MODEL_NAME,
    temperature=0.2
)

In [86]:
class LinkAnalysis(BaseModel):
    """Structured information extracted from a user-provided message."""

    title: str = Field(
        description="A short and accurate title summarizing the message."
    )

    urls: List[str] = Field(
        description="All website, login, and other non-image URLs."
    )

    images: List[str] = Field(
        description="All direct image URLs such as JPG, PNG, GIF, or WEBP."
    )

In [87]:
structured_llm = llm.with_structured_output(LinkAnalysis)

In [88]:
system_prompt = """
You are a URL extraction assistant.

Analyze the user's message and extract:

1. A short, accurate title describing the message.
2. Every URL found in the message.
3. Separate direct image URLs from other URLs.

Rules:
- Do not invent URLs.
- Preserve URLs exactly as they appear.
- A URL containing an image extension such as .jpg, .jpeg, .png,
  .gif, or .webp should be classified as an image.
- Include login pages, redirects, shortened URLs, and other web links
  in the URLs list.
- Do not classify a URL as safe or malicious.
  Security analysis will be performed separately by VirusTotal.
"""

In [89]:
# ==================== YOUR MESSAGE TO ANALYZE ====================
user_message = """
Hey check out these important documents and photos I found:

1. Account update required: http://secure-update-account.net
2. Bank login verification: https://bankofamerica-secure.com/login
3. Apple ID recovery: http://appleid.apple.com.verify-security.info
4. Some photo evidence: https://i.imgur.com/abc123.jpg
5. Another safe link: https://www.google.com
6. PayPal payment confirmation: http://paypal-update-security.com/confirm
7. And this image: https://example.com/report.png
8. One more suspicious one: https://login.microsoftonline.com.phishing-site.net

Thanks!
"""

In [90]:
def get_virustotal_report(url: str) -> dict:
    """
    Retrieve an existing VirusTotal report for a URL.

    The URL is converted to the URL identifier format required
    by the VirusTotal API.
    """

    url_id = base64.urlsafe_b64encode(
        url.encode()
    ).decode().strip("=")

    endpoint = f"https://www.virustotal.com/api/v3/urls/{url_id}"

    headers = {
        "x-apikey": VT_API_KEY,
        "accept": "application/json"
    }

    try:
        response = requests.get(
            endpoint,
            headers=headers,
            timeout=15
        )

        if response.status_code == 200:
            payload = response.json()
            data = payload.get("data", {})
            attributes = data.get("attributes", {})

            stats = attributes.get("last_analysis_stats", {})

            return {
                "status": "found",
                "url": url,
                "malicious": stats.get("malicious", 0),
                "suspicious": stats.get("suspicious", 0),
                "harmless": stats.get("harmless", 0),
                "undetected": stats.get("undetected", 0),
                "timeout": stats.get("timeout", 0),
                "last_analysis_date": attributes.get("last_analysis_date")
            }

        elif response.status_code == 404:
            return {
                "status": "not_found",
                "url": url
            }

        elif response.status_code == 401:
            return {
                "status": "error",
                "url": url,
                "message": "Invalid VirusTotal API key."
            }

        elif response.status_code == 429:
            return {
                "status": "error",
                "url": url,
                "message": "VirusTotal API rate limit exceeded."
            }

        else:
            return {
                "status": "error",
                "url": url,
                "message": f"HTTP {response.status_code}"
            }

    except requests.RequestException as e:
        return {
            "status": "error",
            "url": url,
            "message": str(e)
        }

In [91]:
def get_verdict(result: dict) -> str:
    """
    Generate a simple security verdict from VirusTotal statistics.
    """

    if result["status"] == "not_found":
        return "No VirusTotal report available"

    if result["status"] == "error":
        return "Unable to analyze"

    malicious = result.get("malicious", 0)
    suspicious = result.get("suspicious", 0)

    if malicious > 0:
        return "⚠️ Malicious"

    if suspicious > 0:
        return "⚠️ Suspicious"

    return "ℹ️ No malicious detections reported"

In [92]:
def get_url_risk(result: dict) -> str:
    """Determine the risk level of a single URL."""

    if result.get("status") != "found":
        return "UNKNOWN"

    malicious = result.get("malicious", 0)
    suspicious = result.get("suspicious", 0)

    if malicious >= 3:
        return "HIGH"

    if malicious > 0 or suspicious > 0:
        return "MEDIUM"

    return "LOW"

In [93]:
def get_overall_risk(results: dict) -> str:
    """Determine overall message risk from per-URL risk levels."""

    risks = [
        get_url_risk(result)
        for result in results.values()
    ]

    if "HIGH" in risks:
        return "🔴 HIGH"

    if "MEDIUM" in risks:
        return "🟠 MEDIUM"

    if risks and all(risk == "LOW" for risk in risks):
        return "🟢 LOW"

    return "🟡 UNKNOWN"

In [94]:
def analyze_urls_with_virustotal(urls: list[str]) -> dict:
    """
    Retrieve VirusTotal reports for extracted URLs.
    """

    results = {}

    for url in urls:
        print(f"Checking: {url}")

        results[url] = get_virustotal_report(url)

    return results

In [95]:
def analyze_message(message: str):
    """
    Extract URLs from a message and retrieve their
    VirusTotal security reports.
    """

    chat = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=message)
    ]

    analysis = structured_llm.invoke(chat)

    # Only normal URLs are sent to VirusTotal.
    virustotal_results = analyze_urls_with_virustotal(
        analysis.urls
    )

    return analysis, virustotal_results

In [96]:
def display_analysis_report(analysis, results):
    """Display a complete phishing analysis report."""

    print("=" * 60)
    print("              PHISHING ANALYSIS REPORT")
    print("=" * 60)

    print()
    print(f"Message Title: {analysis.title}")
    print()
    print(f"URLs Detected:    {len(analysis.urls)}")
    print(f"Images Detected:  {len(analysis.images)}")

    print()
    print("-" * 60)
    print("URL ANALYSIS")
    print("-" * 60)

    if not analysis.urls:
        print("\nNo URLs were detected in the message.")

    for index, url in enumerate(analysis.urls, start=1):

        result = results.get(url, {})

        print()
        print(f"{index}. {url}")
        print()

        status = result.get("status")

        if status == "found":

            print("   VirusTotal Report: FOUND")
            print(f"   Malicious:   {result.get('malicious', 0)}")
            print(f"   Suspicious:  {result.get('suspicious', 0)}")
            print(f"   Harmless:    {result.get('harmless', 0)}")
            print(f"   Undetected:  {result.get('undetected', 0)}")
            print(f"   Timeout:     {result.get('timeout', 0)}")
            print()
            print(f"   Verdict: {get_verdict(result)}")

        elif status == "not_found":

            print("   VirusTotal Report: NOT FOUND")
            print("   Verdict: ℹ️ No existing VirusTotal report available")

        else:

            print("   VirusTotal Report: ERROR")
            print(
                f"   Error: {result.get('message', 'Unknown error')}"
            )

        print()
        print("-" * 60)

    print()
    print(f"Overall Risk: {get_overall_risk(results)}")

    print()
    print("=" * 60)

In [97]:
analysis, results = analyze_message(user_message)

display_analysis_report(analysis, results)

Checking: http://secure-update-account.net
Checking: https://bankofamerica-secure.com/login
Checking: http://appleid.apple.com.verify-security.info
Checking: https://www.google.com
Checking: http://paypal-update-security.com/confirm
Checking: https://login.microsoftonline.com.phishing-site.net
              PHISHING ANALYSIS REPORT

Message Title: Important Documents and Photos

URLs Detected:    6
Images Detected:  2

------------------------------------------------------------
URL ANALYSIS
------------------------------------------------------------

1. http://secure-update-account.net

   VirusTotal Report: FOUND
   Malicious:   1
   Suspicious:  0
   Harmless:    55
   Undetected:  36
   Timeout:     0

   Verdict: ⚠️ Malicious

------------------------------------------------------------

2. https://bankofamerica-secure.com/login

   VirusTotal Report: FOUND
   Malicious:   11
   Suspicious:  1
   Harmless:    48
   Undetected:  32
   Timeout:     0

   Verdict: ⚠️ Malicious

----